# einops-repeat-broadcast — worked example 1: Pair every anchor with every ground-truth box

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-repeat-broadcast`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`einops.repeat('a d -> a b d', b=N)` inserts a brand-new axis whose stride is zero, so the data is *viewed* `N` times rather than copied. This is the pair-every-with-every idiom: expand one batch over the count of the other so that the `(A, B, ...)` grid lines up element-for-element. The classic use is object detection, where every anchor box must be compared against every ground-truth box.

## Worked solution

We have `anchors` of shape `(A, 4)` and `gt_boxes` of shape `(B, 4)`, and we want an `(A, B, 4)` pair grid so each anchor sits next to each box.

1. Read off the counts: `A = anchors.shape[0]`, `B = gt_boxes.shape[0]`. We need these as the named axis sizes for `repeat`.
2. Expand the anchors with `repeat(anchors, 'a d -> a b d', b=B)`. The pattern keeps `a` and `d` where they are and slots a new `b` axis in the middle. Because `b` did not exist in the input, `einops` gives it stride 0 — no memory is copied, the same row is revisited `B` times.
3. Expand the boxes the mirror-image way: `repeat(gt_boxes, 'b d -> a b d', a=A)`. Now the new axis is the leading `a`.
4. Both tensors are now `(A, B, 4)` and aligned: index `[i, j]` of the first is anchor `i`, index `[i, j]` of the second is box `j`. Any elementwise op between them now visits every (anchor, box) pair.

In [ ]:
import torch as t
import einops
from einops import repeat

t.manual_seed(0)
anchors = t.randn(5, 4)
gt_boxes = t.randn(7, 4)

def pair_anchors_with_boxes(anchors, gt_boxes):
    A = anchors.shape[0]
    B = gt_boxes.shape[0]
    anchors_b = repeat(anchors, 'a d -> a b d', b=B)
    boxes_b = repeat(gt_boxes, 'b d -> a b d', a=A)
    return anchors_b, boxes_b

ab, bb = pair_anchors_with_boxes(anchors, gt_boxes)
print(ab.shape, bb.shape)
print('shares storage:', ab.data_ptr() == anchors.data_ptr())